In [2]:
import os
import requests
import logging
import time
from dateutil.relativedelta import relativedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
import matplotlib.pyplot as plt
import csv
import pandas as pd 
import numpy as np 
from tqdm.notebook import tqdm
from eutils import EutilsNCBIError, EutilsRequestError
from metapub import PubMedFetcher, pubmedcentral
from datetime import datetime
from tqdm.auto import tqdm
#Import DR module from Functions folder
from Functions import DataRetrieval as DR
#API_KEY
from Reference_files.keys import API_KEY as API_KEY
import urllib
import json

In [ ]:
# Initialize logger
prefix = "test"+str(datetime.now()).split()[0]
file_handler = logging.FileHandler(f"{prefix}_Examples.log", mode='w')
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
file_handler.setFormatter(formatter)
logging.getLogger().addHandler(file_handler)

# From Query to pubmedCentral full text

In [3]:
# Initialize PubMed fetcher
fetcher = PubMedFetcher()

# Query construction from filters

In [ ]:
O = '''("english"[Language] 
NOT "meta-analysis"[Publication Type] 
NOT "review"[Publication Type] 
NOT "retracted publication"[Publication Type] 
NOT "retraction of publication"[Publication Type] 
NOT "published erratum"[Publication Type] 
NOT "controlled clinical trial"[Publication Type] 
NOT "clinical study"[Publication Type] 
NOT "clinical trial"[Publication Type] 
NOT "clinical trial protocol"[Publication Type] 
NOT "clinical trial, phase i"[Publication Type] 
NOT "clinical trial, phase ii"[Publication Type] 
NOT "clinical trial, phase iii"[Publication Type] 
NOT "clinical trial, phase iv"[Publication Type] 
NOT "clinical trial, veterinary"[Publication Type])'''

A1 = '''"nucleoproteins"[MeSH Terms] 
OR "protein interaction mapping"[MeSH Terms] OR ("nucleoprotein"[All Fields] 
OR "nucleoproteins"[All Fields] 
OR "multiprotein"[All Fields] 
OR "multiproteins"[All Fields] 
OR "proteins"[MeSH Terms] 
OR "protein"[All Fields] 
OR "proteins"[All Fields] 
OR "enzyme"[All Fields]) 
AND 
("interact"[All Fields] 
OR "interacted"[All Fields] 
OR "interacting"[All Fields] 
OR "interaction"[All Fields] 
OR "interactions"[All Fields] 
OR "interactivity"[All Fields] 
OR "interacts"[All Fields]) OR ("protein interaction"[All Fields] 
OR "protein interactions"[All Fields] 
OR "interacting protein"[All Fields] 
OR "interacting proteins"[All Fields] OR "multiprotein complexes"[MeSH Terms]) OR
("nucleoprotein"[All Fields] 
OR "nucleoproteins"[All Fields] 
OR "multiprotein"[All Fields] 
OR "multiproteins"[All Fields] 
OR "proteins"[MeSH Terms] 
OR "protein"[All Fields] 
OR "proteins"[All Fields] 
OR "enzyme"[All Fields]) 
AND 
("complex"[All Fields] 
OR "complexes"[All Fields] 
OR "heteromer"[All Fields] 
OR "heteromers"[All Fields] 
OR "homomer"[All Fields] 
OR "homomers"[All Fields] 
OR "heteromeric"[All Fields] 
OR "homomeric"[All Fields] 
OR "subunit"[All Fields] 
OR "subunits"[All Fields])
OR "protein complex"[All Fields] 
OR "protein complexes"[All Fields] OR ("protein"[All Fields] 
AND ("RNA"[All Fields] 
OR "DNA"[All Fields] 
OR "ribonucleic"[All Fields] 
OR "deoxyribonucleic"[All Fields]) 
AND ("interaction"[All Fields] 
OR "interactions"[All Fields] ))'''

A2 = '''"Immunoprecipitation"[Mesh] 
OR coimmunoprecipitation 
OR ("co"[All Fields] AND "immunoprecipitation"[All Fields]) 
OR ("RNA" [All Fields] AND "immunoprecipitation" [All Fields]) 
OR "co immunoprecipitation"[All Fields] OR "coIP"[All Fields] OR "Chromatography, Affinity"[Mesh] OR "affinity purification"[All Fields] OR "affinity isolation"[All Fields] OR "affinity chromatography"[All Fields] 
OR ("affinity"[All Fields] AND ("purification"[All Fields] OR "isolation"[All Fields] OR "chromatography"[All Fields])) 
OR("pull"[All Fields] AND ("down"[All Fields] OR "downs"[All Fields]))'''


A3 = '''"crystallography, x ray"[MeSH Terms]
OR "Nuclear Magnetic Resonance, Biomolecular"[MeSH Terms]
OR "Cryoelectron Microscopy"[MeSH Terms]
OR "Protein Array Analysis"[MeSH Terms]
OR "electrophoretic mobility shift assay"[MeSH Terms] 
OR "Surface Plasmon Resonance"[MeSH Terms]'''

B = '''"Epitope Mapping"[MeSH Terms] 
OR "Precipitin Tests"[MeSH Terms] 
OR "Two-Hybrid System Techniques"[MeSH Terms] 
OR "blotting, far western"[MeSH Terms] 
OR "Radioimmunoprecipitation Assay"[MeSH Terms] 
OR "Autoantibodies"[MeSH Terms] 
OR "Chromatin Immunoprecipitation"[MeSH Terms] 
OR "Cross-Linking Reagents"[MeSH Terms] 
OR "Formaldehyde"[MeSH Terms] 
OR "Microscopy"[MeSH Terms]OR (crosslink AND reagent) 
OR "Formaldehyde"[All Fields] OR "DNA Footprinting"[Mesh] 
OR "Nuclease Protection Assays"[Mesh]  
OR "Blotting, Southwestern"[Mesh] 
OR "Fluorescence Resonance Energy Transfer"[Mesh] 
OR "PAR CLIP"[All Fields]
OR "AlphaFold"[All Fields]'''



query = f"({O} AND ({A1} OR {A3})) NOT ({A1} AND {B} NOT ({A2} AND {A3}))"

In [ ]:
query_file = "./Reference_files/query.txt" # in case query is in a file
query = DR.read_query_from_file(query_file)

In [ ]:
start_date = "2025-01-01"
stop_date = None  # Will default to the current date if None\
pmid_list, intervals = DR.fetch_pmids_over_period(query, start=start_date, stop=stop_date, full_text=False, plot=True, max_workers=2)

In [ ]:
pmc_id_list = DR.get_pmcid_for_otherid(pmid_list)

In [ ]:
oa_pmcids = DR.filter_oa_database(pmc_id_list) # downloads open access database file from NCBI server \size: ~230 mgb\

In [ ]:
## Fetch openaccess PMCs from a query
start_date = "2000-01-01"
stop_date = None  # Will default to the current date if None
pmid_array = DR.fetch_pmids_over_period(query, start=start_date, stop_date=stop_date, full_text = True)
pmc_id_list = DR.get_pmcid_for_otherid(pmid_list)
oa_pmcids = DR.filter_oa_database(pmc_id_list) # downloads open access database file from NCBI server \size: ~230 mgb\

## Download papers using python requests

In [6]:
oa_pmcids = ["PMC9770967"]

In [7]:
success_count, failed = DR.download_pmc_articles(oa_pmcids)


Download Summary:
Successfully processed 1/1 full-text files


###  Or download using powershell \windows/faster\ 

In [ ]:
# Save the oa_pmcids pandas series to a text file that powershell can read(one ID per line)
mkdir -p ./Full_text_jsons

# Read PMCIDs from the file and process each one
Get-Content oa_pmcids.txt | ForEach-Object {
    $pmcid = $_

    # 1. Download full-text JSON
    $jsonUrl = "https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/$pmcid/unicode"
    $jsonOut = "./Full_text_jsons/$pmcid.json"
    Invoke-RestMethod -Uri $jsonUrl -OutFile $jsonOut

    # 2. Download supplementary ZIP
    $zipUrl = "https://www.ebi.ac.uk/europepmc/webservices/rest/$pmcid/supplementaryFiles"
    $zipOut = "./Full_text_jsons/${pmcid}_supp.zip"
    Invoke-WebRequest -Uri $zipUrl -OutFile $zipOut
}

# Or Using bash

In [ ]:
%%bash

mkdir -p ./Full_text_jsons

while IFS= read -r PMCID; do
    echo "Processing $PMCID"

    # 1. Download full-text JSON
    json_url="https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/${PMCID}/unicode"
    curl -s "${json_url}" -o "./Full_text_jsons/${PMCID}.json"

    # 2. Download supplementary ZIP
    zip_url="https://www.ebi.ac.uk/europepmc/webservices/rest/${PMCID}/supplementaryFiles"
    curl -s "${zip_url}" -o "./Full_text_jsons/${PMCID}_supp.zip"
done < full_text_pmc.txt


# Paper Metadata

In [ ]:
paper_meta = DR.fetch_articles_meta(pmid_list)

### add publisher if needed using Xreff

In [ ]:
added_publishers = DR.process_publishers(paper_meta, email="your_email@rcf.edu")

In [ ]:
import random 
pmid_list = pmid_list.tolist()
random_ppers = random.sample(pmid_list, 50)

In [ ]:
paper_meta_50 = DR.fetch_articles_meta(random_ppers)

# process full text json and parse relevant data into a dataframe 

In [8]:
section_types = ['TITLE', 'ABSTRACT', 'INTRO', 'METHODS', 'RESULTS', 'CONCL', 'FIG', 'SUPPL', 'DISCUSS']
full_text_df = DR.extract_text_from_json_to_dataframe("./Full_text_jsons", section_types)

2025-05-28 18:50:12 LAPTOP-S8N3C7A8 Functions.DataRetrieval[21192] INFO Extracting from JSON files in ./Full_text_jsons
2025-05-28 18:50:12 LAPTOP-S8N3C7A8 Functions.DataRetrieval[21192] INFO Processed 1 files, 0 errors
2025-05-28 18:50:12 LAPTOP-S8N3C7A8 Functions.DataRetrieval[21192] INFO Extracted 100 entries


### fetch aditional metadata that is not avalabble in the fulltext json eg. Journal, ISSN

In [9]:
meta_text = DR.add_metadata_to_dataframe(full_text_df)

Fetching metadata:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching metadata: 100%|██████████| 1/1 [00:01<00:00,  1.08s/it]


# BioC supplementary

# Creat full_text table + supplementary 

In [ ]:
full_text_df = DR.extract_text_from_json_to_dataframe("./XML+JSON", section_types, "./XML+JSON")